# Camera Calibration — Module B (Learn OpenCV Step-by-Step)

Is notebook me hum **Module B** ka code chala ke OpenCV camera calibration seekhenge.

## Ye notebook me kya seekhoge
1. **Image ek NumPy array hai** — pixels = grid of numbers (0=black, 255=white)
2. **Checkerboard corners detect karna** — `cv2.findChessboardCornersSB`
3. **Intrinsic matrix K recovery** — `cv2.calibrateCamera`
4. **Lens distortion remove** — `cv2.undistort`

> **Note:** Images tabhi chaleinngi jab `assets/checkerboard/` me photos honge.
> Abhi bas setup chalao aur steps samjho.


In [4]:
# ========================
# SETUP — Libraries import
# ========================
import argparse, glob, json, os
import cv2
import numpy as np

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)

# Project se relative paths
BASE = "."
ASSETS_CALIB = os.path.join("assets", "checkerboard")
print("Checkerboard photos folder:", ASSETS_CALIB)
print("Existing kalibration files:", glob.glob(os.path.join(ASSETS_CALIB, "*")))


ModuleNotFoundError: No module named 'cv2'

## Concept 1 — Image ek NumPy array hai

Tumhare liye sabse important baat: **OpenCV me image ek 2D/3D NumPy array hai.**
- Grayscale image = `(height, width)` → har pixel 0(black) se 255(white)
- Color image = `(height, width, 3)` → 3 channel (B, G, R)

Bina photos ke bhi hum ek fake image bana kar dekh sakte hain.


In [ ]:
# Ek black image banao (200 rows x 300 cols)
fake = np.zeros((200, 300), np.uint8)
print("Shape (rows, cols):", fake.shape)
print("Har pixel ki value:", fake[0, 0])   # 0 = black

# Checkerboard pattern manually banao (jaisa make_checkerboard.py karta hai)
for r in range(0, 200, 50):
    for c in range(0, 300, 50):
        if ((r // 50) + (c // 50)) % 2 == 0:
            fake[r:r+50, c:c+50] = 255      # white block
print("White block ke andar value:", fake[10, 10])   # 255 = white

cv2.imwrite(os.path.join("assets", "fake_checker.png"), fake)
print("Demo checkerboard save ho gaya -> assets/fake_checker.png")
# Display ke liye (Jupyter me inline show hoga)
from IPython.display import Image, display
display(Image(filename=os.path.join("assets", "fake_checker.png")))


## Concept 2 — Photos load karo

`calibrate.py` me `load_images()` sab photos ko ek folder se padhta hai.
Yahi logic chala ke dekho (agar photos na ho to warning aayegi).


In [ ]:
# calibrate.py se copy kiya hua load_images()
# NOTE: board.png ko jaan-boojh kar skip karte hain. Wo PRINT karne wali board
# hai, photo nahi - uske corners to dhundh hi jayenge (wo bilkul flat hai),
# aur wo calibration me galat K de dega. calibrate.py bhi yahi karta hai.
def load_images(image_dir):
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp")
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(image_dir, ext)))
    return sorted(p for p in paths if os.path.basename(p) != "board.png")

paths = load_images(ASSETS_CALIB)
print(f"Assets/checkerboard me {len(paths)} photos mili.")
for p in paths:
    print("  -", os.path.basename(p))

if not paths:
    print("NOTE: Abhi photos nahi hain. Board print karke 15-20 photos dalo.")


## Concept 3 — Checkerboard corners detect karna

Camera calibration ka pehla kaam: har photo me checkerboard ke **interior corners** dhundo.
`cv2.findChessboardCornersSB` ye kaam karta hai (SB = more robust version).

> Board pattern: 9 cols × 6 rows squares → **8 × 5 interior corners**


In [ ]:
# calibrate.py ke arguments
cols, rows = 9, 6
interior = (cols - 1, rows - 1)     # jaise calibrate.py line 66 me hai
square_mm = 24.0
print("Interior corners pattern:", interior, "(8x5)")

# Object points — real world coordinates (in meters), jaise calibrate.py line 73-75
objp = np.zeros((interior[0] * interior[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:interior[0], 0:interior[1]].T.reshape(-1, 2)
objp *= square_mm / 1000.0
print("Object points (first 3):",
      [tuple(np.round(p, 4)) for p in objp[:3]])
print("Ye points checkerboard ke squares ka real-world grid hain.")


## Concept 4 — Real photos me corners dhundo

Agar photos hain to is cell me har photo ke corners detect honge.
`cv2.findChessboardCornersSB` image ko padh kar corners ki (x,y) positions deta hai.


In [ ]:
def find_corners(path, pattern):
    img = cv2.imread(path)
    if img is None:
        return False, None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if hasattr(cv2, "findChessboardCornersSB"):
        ok, corners = cv2.findChessboardCornersSB(gray, pattern, None)
    else:
        ok, corners = cv2.findChessboardCorners(gray, pattern, None)
    return bool(ok), corners

obj_points, img_points, img_shapes, used_paths = [], [], [], []
for path in paths:
    ok, corners = find_corners(path, interior)
    if not ok or corners is None:
        print(f"[!] skip {os.path.basename(path)}: corners NOT detected")
        continue
    img = cv2.imread(path)
    obj_points.append(objp)
    img_points.append(corners.reshape(-1, 2))
    img_shapes.append((img.shape[1], img.shape[0]))
    used_paths.append(path)
    print(f"[+] {os.path.basename(path)}: {len(corners)} corners")

n_good = len(obj_points)
print(f"\nSukseful images: {n_good} (kam se kam 3 chahiye)")


## Concept 5 — cv2.calibrateCamera (K matrix)

Corners mil gaye to `cv2.calibrateCamera` dono (object points + image points) ko use karke:
- **K matrix** = focal length (fx, fy) + optical center (cx, cy)
- **dist** = lens distortion coefficients

Yahi tumhare moduleB ka core hai.


In [ ]:
if n_good >= 3:
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        obj_points, img_points, img_shapes[0], None, None)
    print(f"RMS re-projection error: {rms:.4f} px  (aim < 1.0)")
    print("Intrinsic matrix K:")
    print(np.round(K, 4))
    print("Distortion (k1,k2,p1,p2,k3):", np.round(dist.flatten(), 6).tolist())

    # Save (jaise calibrate.py me)
    data = {
        "pattern": {"squares_cols": cols, "squares_rows": rows},
        "square_mm": square_mm,
        "image_size": {"width": img_shapes[0][0], "height": img_shapes[0][1]},
        "rms": float(rms),
        "K": K.tolist(),
        "dist": dist.flatten().tolist(),
        "n_images": n_good,
        "extrinsics": {os.path.basename(p): {"rvec": r.tolist(), "tvec": t.tolist()}
                       for p, r, t in zip(used_paths, rvecs, tvecs)},
    }
    with open("moduleB/calibration.json", "w") as f:
        json.dump(data, f, indent=2)
    print("\nSaved -> moduleB/calibration.json")
else:
    print("Photos nahi hain, isliye calibrate nahi ho paya.")
    print("Pehle board print karke 15-20 photos daalo.")


## Concept 6 — cv2.undistort (distortion hatana)

Lens image ko curve kar deta hai (edge pe). K matrix + dist se
`cv2.undistort` us effect ko thik karta hai.


In [ ]:
if n_good >= 3 and used_paths:
    p = used_paths[0]
    img = cv2.imread(p)
    h, w = img.shape[:2]
    newK, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1, (w, h))
    und = cv2.undistort(img, K, dist, None, newK)

    # Before | After side by side
    combo = np.hstack([img, und])
    cv2.imwrite("assets/undistort_demo.png", combo)
    print("Before | After demo -> assets/undistort_demo.png")
    try:
        from IPython.display import Image, display
        display(Image(filename="assets/undistort_demo.png"))
    except Exception as e:
        print("(display error):", e)
else:
    print("Pehle photos daalo phir undistort demo dekho.")
